# SEN2SR — original S2 only, tiny patch, save PNGs

**Data source:** only `SDS_performance_analysis/.../sat_images` originals (paths containing `S2`). **Not** bilinear/bicubic outputs from this repo.

**Why two steps before the model:**

1. **LR-only cell** — reads a **128×128** center crop with rasterio, saves `opensr_outputs/lr_crop_only.png`. If that works, I/O + band logic are fine.
2. **Model cell** — loads weights.
3. **SR cell** — one **direct** `model(batch)` forward on the same 128×128 tensor (**no `sen2sr.predict_large`**), then saves `opensr_outputs/sen2sr_lr_vs_sr.png`. The tiled `predict_large` helper has been a common source of hangs/kernel crashes after tqdm finishes.

Uses **Agg** + **`savefig` / `plt.close()`** so the notebook does not need to render large canvases.

```bash
pip install sen2sr mlstac torch rasterio matplotlib numpy
```

Run **all** code cells top to bottom after a kernel restart.

In [ ]:
# %pip install sen2sr mlstac torch rasterio matplotlib numpy

In [1]:
from __future__ import annotations

import warnings
from pathlib import Path
from typing import Dict, List, Tuple

import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import rasterio
from rasterio.windows import Window
import torch

import mlstac


def _normalize_name(name: str) -> str:
    return "".join(ch for ch in name.lower() if ch.isalnum())


def _band_aliases() -> Dict[str, Tuple[str, ...]]:
    return {
        "blue": ("blue", "b", "band2", "b2"),
        "green": ("green", "g", "band3", "b3"),
        "red": ("red", "r", "band4", "b4"),
        "nir": ("nir", "nearinfrared", "nearir", "band5", "b5", "nir08", "nir1"),
    }


def extract_band_names(src: rasterio.io.DatasetReader) -> List[str]:
    names: List[str] = []
    for i in range(1, src.count + 1):
        desc = src.descriptions[i - 1] if src.descriptions else None
        tags = src.tags(i)
        tag_name = tags.get("name") or tags.get("band_name") or tags.get("long_name")
        chosen = (desc or tag_name or f"band{i}").strip()
        names.append(chosen)
    return names


def find_band_index(band_names: List[str], key: str) -> int:
    aliases = tuple(_normalize_name(a) for a in _band_aliases()[key])
    normalized = [_normalize_name(n) for n in band_names]
    for idx, n in enumerate(normalized):
        if n in aliases:
            return idx
    for idx, n in enumerate(normalized):
        if any(a in n for a in aliases):
            return idx
    raise ValueError(f"Could not find band '{key}' in names: {band_names}")


def to_reflectance_physical(data: np.ndarray) -> np.ndarray:
    mx = float(np.nanmax(data))
    if mx > 10:
        return (data / 10000.0).astype(np.float32)
    return np.clip(data, 0.0, 1.0).astype(np.float32)


def read_rgbn_center_patch(path: Path, patch: int = 128) -> np.ndarray:
    """Center-crop (patch×patch) RGB+NIR as (4, patch, patch) reflectance float32. Zero-pad if image smaller than patch."""
    with rasterio.open(path) as src:
        if src.count < 4:
            raise ValueError(f"Need ≥4 bands, got {src.count} for {path}")
        h, w = src.height, src.width
        side = min(patch, h, w)
        row_off = max(0, (h - side) // 2)
        col_off = max(0, (w - side) // 2)
        window = Window(col_off, row_off, side, side)
        data = src.read(window=window).astype(np.float32)
        names = extract_band_names(src)

    ri = find_band_index(names, "red")
    gi = find_band_index(names, "green")
    bi = find_band_index(names, "blue")
    ni = find_band_index(names, "nir")
    stacked = np.stack([data[ri], data[gi], data[bi], data[ni]], axis=0)
    stacked = to_reflectance_physical(stacked)

    if side < patch:
        pad_h = patch - stacked.shape[1]
        pad_w = patch - stacked.shape[2]
        stacked = np.pad(
            stacked,
            ((0, 0), (0, pad_h), (0, pad_w)),
            mode="constant",
            constant_values=0.0,
        )
    return stacked


def rgb_composite(stack_4: np.ndarray) -> np.ndarray:
    r, g, b = stack_4[0], stack_4[1], stack_4[2]
    rgb = np.stack([r, g, b], axis=-1).astype(np.float64)
    out = np.zeros_like(rgb)
    for c in range(3):
        v = rgb[..., c]
        finite = np.isfinite(v)
        if not finite.any():
            continue
        lo, hi = np.percentile(v[finite], (2.0, 98.0))
        if hi <= lo:
            hi = lo + 1e-6
        out[..., c] = np.clip((v - lo) / (hi - lo), 0.0, 1.0)
    return out.astype(np.float32)

In [2]:
try:
    PROJECT_ROOT = Path(__file__).resolve().parent
except NameError:
    PROJECT_ROOT = Path.cwd()

# Original imagery only (not change_tiff_resolution/data/sat_images upsamples).
original_root = (
    PROJECT_ROOT.parent / "SDS_performance_analysis" / "data" / "sat_images"
)

MODEL_DIR = PROJECT_ROOT / "opensr_weights" / "SEN2SRLite_RGBN"
HF_MLM = "https://huggingface.co/tacofoundation/sen2sr/resolve/main/SEN2SRLite/NonReference_RGBN_x4/mlm.json"

OUTPUT_DIR = PROJECT_ROOT / "opensr_outputs"
LR_PREVIEW_PNG = OUTPUT_DIR / "lr_crop_original_only.png"
SR_COMPARE_PNG = OUTPUT_DIR / "sen2sr_lr_vs_sr.png"

LR_PATCH = 128  # native LR pixels for one SEN2SR forward (no tiling helper)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


def pick_one_s2_geotiff(root: Path) -> Path:
    cands = sorted(root.glob("**/S2/**/*.tif")) + sorted(root.glob("**/S2/**/*.tiff"))
    for p in cands:
        if p.is_file():
            return p
    raise FileNotFoundError(f"No S2 GeoTIFF under {root}")


scene_path = pick_one_s2_geotiff(original_root)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("PROJECT_ROOT  :", PROJECT_ROOT.resolve())
print("original_root :", original_root.resolve())
print("scene_path    :", scene_path)
print("LR_PATCH      :", LR_PATCH)
print("device        :", device)
print("CUDA available:", torch.cuda.is_available())
print("LR preview PNG:", LR_PREVIEW_PNG.resolve())
print("SR compare PNG :", SR_COMPARE_PNG.resolve())

PROJECT_ROOT  : C:\Users\jnicolow\Documents\research\CRC\change_tiff_resolution
original_root : C:\Users\jnicolow\Documents\research\CRC\SDS_performance_analysis\data\sat_images
scene_path    : c:\Users\jnicolow\Documents\research\CRC\SDS_performance_analysis\data\sat_images\australianarrabeen\S2\S2_20151218_235302.tif
LR_PATCH      : 128
device        : cuda
CUDA available: True
LR preview PNG: C:\Users\jnicolow\Documents\research\CRC\change_tiff_resolution\opensr_outputs\lr_crop_original_only.png
SR compare PNG : C:\Users\jnicolow\Documents\research\CRC\change_tiff_resolution\opensr_outputs\sen2sr_lr_vs_sr.png


In [3]:
# --- Step 1: rasterio only — tiny crop from ORIGINAL SDS GeoTIFF, save LR PNG ---
lr_stack = read_rgbn_center_patch(scene_path, patch=LR_PATCH)
print("LR stack (C,H,W):", lr_stack.shape)

fig, ax = plt.subplots(1, 1, figsize=(4, 4))
ax.imshow(rgb_composite(lr_stack), interpolation="nearest")
ax.set_title(f"Original S2 crop\n{LR_PATCH}×{LR_PATCH} px @ ~10 m")
ax.axis("off")
fig.tight_layout()
fig.savefig(LR_PREVIEW_PNG, dpi=120, bbox_inches="tight")
plt.close(fig)

print("Wrote LR preview:", LR_PREVIEW_PNG.resolve())

LR stack (C,H,W): (4, 128, 128)
Wrote LR preview: C:\Users\jnicolow\Documents\research\CRC\change_tiff_resolution\opensr_outputs\lr_crop_original_only.png


In [4]:
MODEL_DIR.mkdir(parents=True, exist_ok=True)
mlm_path = MODEL_DIR / "mlm.json"

if not mlm_path.exists():
    print("Downloading SEN2SRLite (NonReference RGBN ×4) …")
    loader = mlstac.download(file=HF_MLM, output_dir=MODEL_DIR)
else:
    loader = mlstac.load(mlm_path.as_posix())

model = loader.compiled_model(device=device)
model.eval()
print("Model ready on", device)

Model ready on cuda


In [6]:
# --- Step 2: single 128→SR forward (no predict_large tiling) ---
X = torch.from_numpy(lr_stack).float().to(device)
X = torch.nan_to_num(X, nan=0.0, posinf=0.0, neginf=0.0)

print("Running single model forward …", flush=True)
with torch.inference_mode():
    out = model(X.unsqueeze(0)).squeeze(0)

if torch.cuda.is_available():
    torch.cuda.synchronize()

sr = out.detach().float().cpu().numpy()
del out, X
if torch.cuda.is_available():
    torch.cuda.empty_cache()

print("SR shape (C,H,W):", sr.shape, flush=True)

lr_rgb = rgb_composite(lr_stack)
sr_rgb = rgb_composite(sr)

fig, axes = plt.subplots(1, 2, figsize=(10, 4.5))
axes[0].imshow(lr_rgb, interpolation="nearest")
axes[0].set_title(f"LR original crop\n{LR_PATCH}×{LR_PATCH} @ ~10 m")
axes[0].axis("off")
axes[1].imshow(sr_rgb, interpolation="nearest")
axes[1].set_title(f"SEN2SR output\n{sr_rgb.shape[0]}×{sr_rgb.shape[1]} px")
axes[1].axis("off")
fig.suptitle(scene_path.name, fontsize=10)
fig.tight_layout()
fig.savefig(SR_COMPARE_PNG, dpi=140, bbox_inches="tight")
plt.close(fig)

print("Wrote comparison:", SR_COMPARE_PNG.resolve(), flush=True)

Running single model forward …
SR shape (C,H,W): (4, 512, 512)
Wrote comparison: C:\Users\jnicolow\Documents\research\CRC\change_tiff_resolution\opensr_outputs\sen2sr_lr_vs_sr.png
